# OSDK 'Hello World'

**실행 전 준비**: `conda activate osdk-hello` → `.\run-jupyter.ps1` 로 실행하세요.
`FOUNDRY_HOSTNAME`(`..\_shared\.env`)과 `FOUNDRY_TOKEN`(이 폴더 `.env`)이 환경변수로 주입됩니다.

객체는 `[Example] Airport`(API name `ExampleAirport`)를 씁니다. SDK 패키지는 `osdk_hello_world_app_sdk`.

이 SDK 버전(2.231.0)에는 `UserTokenAuth`가 없어서, 고정 토큰을 그대로 돌려주는 `Auth`/`Token` 구현체를 직접 만들어 씁니다.

**남은 TODO 1곳**: primary key. 셀 1을 먼저 돌려 `take(1)` 출력에서 PK를 눈으로 확인한 뒤 셀 2를 채우세요.

In [8]:
import os
from osdk_hello_world_app_sdk import FoundryClient
from foundry_sdk._core.auth_utils import Auth, Token

HOSTNAME = os.environ["FOUNDRY_HOSTNAME"]  # ..\_shared\.env 에서 주입
TOKEN = os.environ["FOUNDRY_TOKEN"]        # .\.env 에서 주입 (셀에 하드코딩 금지)
print("hostname:", HOSTNAME, "| token:", f"{len(TOKEN)}자 주입됨")

class StaticToken(Token):
    def __init__(self, token):
        self._token = token
    @property
    def access_token(self):
        return self._token

class StaticTokenAuth(Auth):
    def __init__(self, token):
        self._token = StaticToken(token)
    def get_token(self):
        return self._token
    def execute_with_token(self, func):
        return func(self._token)
    def run_with_token(self, func):
        func(self._token)

auth = StaticTokenAuth(TOKEN)
client = FoundryClient(auth=auth, hostname=HOSTNAME)

exampleAirportObject = client.ontology.objects.ExampleAirport
print(exampleAirportObject.take(1))

hostname: https://dataartai.usw-23.palantirfoundry.com | token: 311자 주입됨
[ExampleAirport(rid=None, average_dep_delay=7.548666186012977, display_city_market_name_full=Cedar Rapids/Iowa City, IA, airport_state_code=IA, airport_start_date=2011-07-01 12:00:00+00:00, airport_id=11003, arriving_flight_count=1403, latitude=41.88472222, geopoint=coordinates=[-91.7108333, 41.8847222] bbox=None type='Point', complete_flight_history=None, airport_state_name=Iowa, display_airport_city_name_full=Cedar Rapids/Iowa City, IA, airport=CID, average_arr_delay=9.27765726681128, display_airport_name=The Eastern Iowa, departing_flight_count=1403, longitude=-91.71083333, daily_avg_arr_delay=Timeseries(dailyAvgArrDelay), daily_count_of_flights=Timeseries(dailyCountOfFlights), daily_avg_dep_delay=Timeseries(dailyAvgDepDelay))]


In [10]:
# TODO — 위 셀의 take(1) 출력에서 읽은 실제 primary key(Airport Id)로 교체.
primaryKey = "11003"

result = client.ontology.objects.ExampleAirport.get(primaryKey)

# 속성명은 take(1) 출력에서 실제 필드명을 확인해 맞추세요 (display_airport_name 이 아닐 수 있음).
result.display_airport_name

'The Eastern Iowa'